%md
# Phase 4A – Bucketing & Segmentation in PySpark

## Objective

Learn how to convert continuous values into business categories using PySpark.

Topics covered:

- Conditional Bucketing
- SQL CASE Statement
- Bucketizer (MLlib)
- Quantile-based Segmentation
- Window Ranking

### Step 1: Start Spark Session

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Phase4A_Bucketing") \
    .getOrCreate()

### Step 2: Read Datasets

In [0]:
customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/databricks2027/customers.csv")

sales = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/databricks2027/sales.csv")

display(customers)
display(sales)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701
2,Emma,Jones,emma.jones@webmail.com,555-0002,456 Oak St,Centerville,OH,45459
3,Olivia,Brown,olivia.brown@outlook.com,555-0003,789 Pine St,Greenville,SC,29601
4,Liam,Johnson,liam.johnson@gmail.com,555-0004,101 Maple St,Riverside,CA,92501
5,Noah,Williams,noah.williams@yahoo.com,555-0005,202 Birch St,Lakeside,TX,75001
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601
7,Isabella,Davis,isabella.davis@icloud.com,555-0007,404 Spruce St,Boise,ID,83701
8,James,Martinez,james.martinez@live.com,555-0008,505 Walnut St,Des Moines,IA,50301
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201


sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,1,1,2024-01-15,2,39.98
2,1,3,2024-01-20,1,29.99
3,2,2,2024-01-16,1,25.0
4,2,4,2024-01-22,3,89.97
5,3,5,2024-01-17,2,49.98
6,4,6,2024-01-18,4,119.96
7,4,7,2024-01-25,1,15.5
8,5,8,2024-01-19,3,66.75
9,6,9,2024-01-20,2,40.0
10,7,10,2024-01-21,5,110.95


### Step 3: Clean Data

In [0]:
customers_clean = customers.dropna().dropDuplicates()

sales_clean = sales.dropna().dropDuplicates()

### Step 4: Join Customer and Sales Data

In [0]:
customer_sales = customers_clean.join(
    sales_clean,
    on="customer_id",
    how="inner"
)
display(customer_sales)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code,sale_id,product_id,sale_date,quantity,total_amount
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601,9,9,2024-01-20,2,40.0
31,Chloe,Adams,chloe.adams@aol.com,555-0031,2828 Elm St,San Jose,CA,95101,35,15,2024-02-18,2,50.0
33,Grace,Baker,grace.baker@live.com,555-0033,3030 Cedar St,Jackson,MS,39201,37,17,2024-02-20,2,34.0
40,Benjamin,Evans,benjamin.evans@zoho.com,555-0040,3737 Elm St,Denver,CO,80202,44,4,2024-02-27,2,40.0
43,Lily,Turner,lily.turner@webmail.com,555-0043,4040 Spruce St,Chicago,IL,60601,47,7,2024-03-02,2,40.0
44,Daniel,Morris,daniel.morris@zoho.com,555-0044,4141 Walnut St,San Diego,CA,92102,48,8,2024-03-03,3,55.5
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701,2,3,2024-01-20,1,29.99
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201,12,12,2024-01-23,4,79.96
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201,13,13,2024-01-24,2,55.0
15,Harper,Jackson,harper.jackson@outlook.com,555-0015,1212 Cedar St,Seattle,WA,98101,18,18,2024-01-29,4,92.0


%md
## Practice Task 1

### Create Gold / Silver / Bronze Segmentation using Conditional Logic

In [0]:
customer_sales.createOrReplaceTempView("customer_sales")

- SQL

In [0]:
%sql

SELECT customer_id,total_amount,
CASE
    WHEN total_amount > 10000 THEN 'Gold'
    WHEN total_amount BETWEEN 5000 AND 10000 THEN 'Silver'
    ELSE 'Bronze'
END AS segment

FROM customer_sales;

customer_id,total_amount,segment
4,15.5,Bronze
19,29.99,Bronze
22,119.96,Bronze
43,40.0,Bronze
1,39.98,Bronze
26,66.75,Bronze
2,89.97,Bronze
13,34.0,Bronze
18,19.99,Bronze
18,39.98,Bronze


- PySpark

In [0]:
from pyspark.sql.functions import when

segment_df = customer_sales.withColumn(
    "segment",
    when(customer_sales.total_amount > 10000, "Gold")
    .when(
        (customer_sales.total_amount >= 5000) &
        (customer_sales.total_amount <= 10000),
        "Silver"
    )
    .otherwise("Bronze")
)

display(segment_df)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code,sale_id,product_id,sale_date,quantity,total_amount,segment
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601,9,9,2024-01-20,2,40.0,Bronze
31,Chloe,Adams,chloe.adams@aol.com,555-0031,2828 Elm St,San Jose,CA,95101,35,15,2024-02-18,2,50.0,Bronze
33,Grace,Baker,grace.baker@live.com,555-0033,3030 Cedar St,Jackson,MS,39201,37,17,2024-02-20,2,34.0,Bronze
40,Benjamin,Evans,benjamin.evans@zoho.com,555-0040,3737 Elm St,Denver,CO,80202,44,4,2024-02-27,2,40.0,Bronze
43,Lily,Turner,lily.turner@webmail.com,555-0043,4040 Spruce St,Chicago,IL,60601,47,7,2024-03-02,2,40.0,Bronze
44,Daniel,Morris,daniel.morris@zoho.com,555-0044,4141 Walnut St,San Diego,CA,92102,48,8,2024-03-03,3,55.5,Bronze
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701,2,3,2024-01-20,1,29.99,Bronze
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201,12,12,2024-01-23,4,79.96,Bronze
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201,13,13,2024-01-24,2,55.0,Bronze
15,Harper,Jackson,harper.jackson@outlook.com,555-0015,1212 Cedar St,Seattle,WA,98101,18,18,2024-01-29,4,92.0,Bronze


%md
## Practice Task 2

### Group Data by Segment and Count Customers

In [0]:
%sql

SELECT
    segment,
    COUNT(customer_id) AS total_customers
FROM (
    SELECT
        customer_id,
        CASE
            WHEN total_amount > 10000 THEN 'Gold'
            WHEN total_amount BETWEEN 5000 AND 10000 THEN 'Silver'
            ELSE 'Bronze'
        END AS segment
    FROM customer_sales
)
GROUP BY segment
ORDER BY total_customers DESC;

segment,total_customers
Bronze,50


In [0]:
segment_count = segment_df.groupBy("segment") \
    .count() \
    .orderBy("count", ascending=False)

display(segment_count)

segment,count
Bronze,50


%md
## Practice Task 3

### Try Quantile-Based Segmentation

In [0]:
from pyspark.sql.functions import when

# Calculate quantiles
quantiles = customer_sales.approxQuantile(
    "total_amount",
    [0.33, 0.66],
    0
)

print("Quantiles:", quantiles)

q1 = quantiles[0]
q2 = quantiles[1]

# Create quantile-based segments
quantile_segment = customer_sales.withColumn(
    "quantile_segment",
    when(customer_sales.total_amount <= q1, "Bronze")
    .when(customer_sales.total_amount <= q2, "Silver")
    .otherwise("Gold")
)

display(quantile_segment)

Quantiles: [35.98, 60.0]


customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code,sale_id,product_id,sale_date,quantity,total_amount,quantile_segment
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601,9,9,2024-01-20,2,40.0,Silver
31,Chloe,Adams,chloe.adams@aol.com,555-0031,2828 Elm St,San Jose,CA,95101,35,15,2024-02-18,2,50.0,Silver
33,Grace,Baker,grace.baker@live.com,555-0033,3030 Cedar St,Jackson,MS,39201,37,17,2024-02-20,2,34.0,Bronze
40,Benjamin,Evans,benjamin.evans@zoho.com,555-0040,3737 Elm St,Denver,CO,80202,44,4,2024-02-27,2,40.0,Silver
43,Lily,Turner,lily.turner@webmail.com,555-0043,4040 Spruce St,Chicago,IL,60601,47,7,2024-03-02,2,40.0,Silver
44,Daniel,Morris,daniel.morris@zoho.com,555-0044,4141 Walnut St,San Diego,CA,92102,48,8,2024-03-03,3,55.5,Silver
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701,2,3,2024-01-20,1,29.99,Bronze
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201,12,12,2024-01-23,4,79.96,Gold
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201,13,13,2024-01-24,2,55.0,Silver
15,Harper,Jackson,harper.jackson@outlook.com,555-0015,1212 Cedar St,Seattle,WA,98101,18,18,2024-01-29,4,92.0,Gold


%md
## Practice Task 4

### Compare Results of Different Methods

In [0]:
comparison = segment_df.join(
    quantile_segment.select(
        "sale_id",
        "quantile_segment"
    ),
    on="sale_id"
)

display(
    comparison.select(
        "customer_id",
        "total_amount",
        "segment",
        "quantile_segment"
    )
)

customer_id,total_amount,segment,quantile_segment
4,15.5,Bronze,Bronze
19,29.99,Bronze,Bronze
22,119.96,Bronze,Gold
43,40.0,Bronze,Silver
1,39.98,Bronze,Silver
26,66.75,Bronze,Gold
2,89.97,Bronze,Gold
13,34.0,Bronze,Bronze
18,19.99,Bronze,Bronze
18,39.98,Bronze,Silver


%md
## Practice Task 5

### Reflection

### Which method is most useful and why?

Conditional Logic (Gold/Silver/Bronze) is most useful when businesses have predefined rules for customer classification, such as loyalty programs or membership levels.

Quantile-based segmentation is useful when the goal is to divide customers into balanced groups based on the data distribution. It automatically adapts to changing customer spending patterns.

In real-world business applications, conditional logic is generally preferred because it is simple, easy to understand, and aligns with business policies. Quantile-based segmentation is more suitable for data analysis, customer analytics, and machine learning applications where balanced groups are important.